# Build a Data Cleaning Helper

In the EDA lesson, we built a helper that answered questions about data. In this lesson we build one that fixes it. Data cleaning is an important step in the Data Science pipeline and let's see how we can utilise LLMs to do that.



## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas python-dotenv

In [ ]:
import os
import re
import warnings
import pandas as pd
from google import genai
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

df = pd.read_csv("../../data/hr_analytics.csv")
print(f"Dataset: {df.shape}")

Over view of the cleaning helper

![](../../images/cleaning_helper.png)

## 2 - The Data

**Some real problems in this dataset**

| Problem | Column(s) | Why it matters |
|---|---|---|
| Missing values | `manager_rating`, `distance_from_home`, `training_hours_last_year` | Imputation policy affects downstream modeling |
| Mixed date formats | `last_promotion_date` | Inconsistent parsing creates silent bugs |

In [ ]:
# Missingness snapshot
df.isnull().sum()

In [ ]:
# Mixed date formats
df["last_promotion_date"]

## 3 - Build the Profile

We will compute a lean statistical summary of the table so that the model gets a clearer view of what actually appears in the column.

In [ ]:
def build_dataframe_profile(frame: pd.DataFrame) -> pd.DataFrame:
    """Build a compact profile of each column.

    Args:
        frame: Input DataFrame. Not modified in place.

    Returns:
        DataFrame with one row per column.
    """
    missing = frame.isna().sum()
    rows = []

    for col in frame.columns:
        series = frame[col]

        # top values with counts
        top_values = series.value_counts(dropna=True).head(5)
        top_values = [(str(v), int(c)) for v, c in top_values.items()]

        rows.append(
            {
                "column": col,
                "dtype": str(series.dtype),
                "missing": int(missing[col]),
                "missing_pct": round(missing[col] / len(frame) * 100, 1),
                "unique": int(series.nunique(dropna=True)),
                "top_values": top_values,
            }
        )

    return pd.DataFrame(rows)


def profile_to_str(profile_df: pd.DataFrame) -> str:
    """Convert profile DataFrame to a string for the LLM."""
    lines = []

    for _, row in profile_df.iterrows():
        lines.append(
            f"{row['column']} | dtype={row['dtype']} | "
            f"missing={row['missing']} ({row['missing_pct']}%) | "
            f"unique={row['unique']} | "
            f"top_values={row['top_values']}"
        )

    return "\n".join(lines)

In [ ]:
build_dataframe_profile(df)

## 4 - The Cleaning Helper

We define two prompts with two specific jobs. The diagnose step reasons about the data first and this catches more issues than going straight to code. You see the diagnosis before any code runs, so you can review and reject before implementation.
What the LLM does here: reads the profile, identifies every data quality issue, then writes the pandas code to fix them. You stay in control of what runs.

In [ ]:
DIAGNOSE_PROMPT = (
    "You are a data quality expert. "
    "Given a dataset profile, identify every data quality issue you can find. "
    "For each issue state: the column, the problem, and the recommended fix. "
    "Consider: missing values, mixed formats, whitespace, inconsistent casing, "
    "identifier columns that should not be modified. "
    "Never use infer_datetime_format — removed in pandas 2.2. "
    "Return a structured list only. No code."
)

IMPLEMENT_PROMPT = (
    "You are a data-cleaning assistant. "
    "Write executable pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. "
    "For numeric columns with low unique counts (rating scales), prefer mode over median for imputation. "
    "Never use infer_datetime_format — removed in pandas 2.2. "
    "For mixed date formats use pd.to_datetime(df[col], format='mixed'). "
    "Do not use inplace=True. Assign results back to columns. "
    "No imports. Save the final cleaned DataFrame to `result_df`. "
    "Return only executable Python code."
)

In [ ]:
def diagnose_data(frame: pd.DataFrame):
    profile = build_dataframe_profile(frame)
    profile_str = profile_to_str(profile)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=profile_str,
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": DIAGNOSE_PROMPT,
        },
    )
    return (response.text or "").strip()


In [ ]:
diagnosis = diagnose_data(df)
print(diagnosis)

In [ ]:
def generate_cleaning_code(frame: pd.DataFrame, diagnosis: str):
    profile = build_dataframe_profile(frame)
    profile_str = profile_to_str(profile)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"{profile_str}\n\nCleaning plan:\n{diagnosis}",
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": IMPLEMENT_PROMPT,
        },
    )

    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    return match.group(1).strip() if match else text


In [ ]:
code = generate_cleaning_code(df, diagnosis)
print(code)

In [ ]:
def apply_cleaning_code(frame: pd.DataFrame, code: str):
    env = {"pd": pd, "df": frame.copy()}
    exec(code, env, env)
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result


In [ ]:
df_clean = apply_cleaning_code(df, code)
df_clean.head()

In [ ]:
df_clean.isnull().sum()

## 5 - Wrapping all the modules in a single function


In [ ]:
def cleaning_helper(
    frame: pd.DataFrame, show_code: bool = False, show_diagnosis: bool = False
):
    diagnosis = diagnose_data(frame)

    if show_diagnosis:
        print("--- Diagnosis ---")
        print(diagnosis)
        print()

    code = generate_cleaning_code(frame, diagnosis)

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---\n")

    result_df = apply_cleaning_code(frame, code)
    return result_df, diagnosis, code


In [ ]:
# run the cleaning helper
df_clean, diagnosis, code = cleaning_helper(df, show_code=True, show_diagnosis=True)

In [ ]:
# verify — shape preserved, missing values gone
print(f"Missing before: {df.isnull().sum().sum()}")
print(f"Missing after:  {df_clean.isnull().sum().sum()}")
print(f"last_promotion_date dtype: {df_clean['last_promotion_date'].dtype}")

In [ ]:
df_clean["last_promotion_date"].head()

## 6 - Edit and Re-run

If the generated code got something wrong, edit the specific line and re-run. No second LLM call needed for small fixes.
If the issue is bigger than a one-line fix, just re-run the helper with additional instructions appended to the profile



In [ ]:
# Editing LLM-generated code is as simple as a string replace.
# Example: swap any method the model got wrong before running it.
code_fixed = code.replace("<what the LLM wrote>", "<what you actually want>")

# e.g. code.replace("df['OverTime'].str.title()", "df['OverTime'].str.upper()")

env = {"pd": pd, "df": df.copy()}
exec(code_fixed, env, env)
env.get("result_df")

## 7 - Save for Reuse

The helper is saved to `cleaning_helper.py`. Lesson 2.2 imports it directly.


In [ ]:
import inspect

components = [
    "import os",
    "import re",
    "import pandas as pd",
    "from google import genai",
    "from dotenv import load_dotenv",
    "",
    "load_dotenv()",
    "client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))",
    "",
    f"DIAGNOSE_PROMPT = {repr(DIAGNOSE_PROMPT)}",
    "",
    f"IMPLEMENT_PROMPT = {repr(IMPLEMENT_PROMPT)}",
    "",
    inspect.getsource(build_dataframe_profile),
    "",
    inspect.getsource(profile_to_str),
    "",
    inspect.getsource(diagnose_data),
    "",
    inspect.getsource(generate_cleaning_code),
    "",
    inspect.getsource(apply_cleaning_code),
    "",
    inspect.getsource(cleaning_helper),
]

with open("cleaning_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved cleaning_helper.py")
